# 🤖 PENGU Scalping Bot — BingX v3
**4 celdas. Ejecutar en orden. Solo tocar Celda 1.**

### Estrategia
- **LONG**: Stochastic RSI en sobreventa (K < 20) + Cipher B WT1 cruza arriba WT2 desde zona OS
- **SHORT**: Stochastic RSI en sobrecompra (K > 80) + Cipher B WT1 cruza abajo WT2 desde zona OB
- Trailing stop dinámico + TP/SL fijo inicial

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 1 — Setup y configuración                 ║
# ╚══════════════════════════════════════════════════╝
!pip install pandas numpy requests --quiet

import requests, hmac, hashlib, time, threading
import pandas as pd, numpy as np
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')

# ─── COMPLETAR ANTES DE EJECUTAR ───────────────────
API_KEY    = "TU_API_KEY_AQUI"
SECRET_KEY = "TU_SECRET_KEY_AQUI"
# ───────────────────────────────────────────────────

BASE_URL    = "https://open-api.bingx.com"
SYMBOL      = "PENGU-USDT"   # se auto-detecta al iniciar
INTERVAL    = "15m"
LEVERAGE    = 10
CAPITAL_PCT = 10             # % del balance por trade

TP_PCT             = 0.015   # 1.5%
SL_PCT             = 0.007   # 0.7%
TRAIL_ACTIVATE_PCT = 0.010   # trailing activa con +1.0% ganancia
TRAIL_CALLBACK_PCT = 0.004   # trail sigue a 0.4% del pico

# ── Cipher B / WaveTrend ───────────────────────────
WT_N1 = 10      # channel length
WT_N2 = 21      # average length
WT_OB = 53      # overbought (señal SHORT)
WT_OS = -53     # oversold   (señal LONG)

# ── Stochastic RSI ─────────────────────────────────
STOCH_RSI_LEN = 14   # período RSI base
STOCH_LEN     = 14   # período Stochastic
STOCH_K       = 3    # suavizado %K
STOCH_D       = 3    # suavizado %D
STOCH_OB      = 80   # sobrecompra
STOCH_OS      = 20   # sobreventa

# ── General ────────────────────────────────────────
VOL_MULT     = 1.1
COOLDOWN     = 3         # velas de espera post-cierre
KLINES_LIMIT = 150
SLEEP_MAIN   = 60 * 13  # 13 min (vela 15m)
SLEEP_TRAIL  = 30

print('✅ Celda 1 OK')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 2 — API BingX + Datos de mercado          ║
# ╚══════════════════════════════════════════════════╝

def _ts():
    return int(time.time() * 1000)

def _h():
    return {"X-BX-APIKEY": API_KEY}

def _signed_url(path, params):
    # Build query string in insertion order, sign it, then append signature.
    # BingX verifies the signature against the raw query string as sent in the URL,
    # so the signed string and the URL params must use identical ordering.
    p = dict(params)
    p["timestamp"] = _ts()
    qs = "&".join(f"{k}={v}" for k, v in p.items())
    sig = hmac.new(SECRET_KEY.encode(), qs.encode(), hashlib.sha256).hexdigest()
    return f"{BASE_URL}{path}?{qs}&signature={sig}"

def api_get(path, params=None, signed=False):
    try:
        if signed:
            url = _signed_url(path, params or {})
            return requests.get(url, headers=_h(), timeout=10).json()
        return requests.get(BASE_URL + path, params=(params or {}), headers=_h(), timeout=10).json()
    except Exception as e:
        print(f"  ⚠️ GET {path}: {e}")
        return {"code": -1}

def api_post(path, params=None):
    try:
        url = _signed_url(path, params or {})
        return requests.post(url, headers=_h(), timeout=10).json()
    except Exception as e:
        print(f"  ⚠️ POST {path}: {e}")
        return {"code": -1}

def api_delete(path, params=None):
    try:
        url = _signed_url(path, params or {})
        return requests.delete(url, headers=_h(), timeout=10).json()
    except Exception as e:
        print(f"  ⚠️ DELETE {path}: {e}")
        return {"code": -1}

def detect_symbol():
    global SYMBOL
    for sym in ["PENGU-USDT", "PENGUSDT", "PENGU_USDT"]:
        r = api_get("/openApi/swap/v2/quote/klines",
                    {"symbol": sym, "interval": "15m", "limit": 3})
        if r.get("code") == 0 and r.get("data"):
            SYMBOL = sym
            print(f"  ✅ Símbolo: {SYMBOL}")
            return True
    print("  ❌ PENGU no encontrado en BingX Futuros")
    return False

def get_klines():
    r = api_get("/openApi/swap/v2/quote/klines",
                {"symbol": SYMBOL, "interval": INTERVAL, "limit": KLINES_LIMIT})
    if not r or r.get("code") != 0:
        print(f"  ❌ Klines: {r.get('msg', r)}")
        return None
    raw = r.get("data", [])
    if not raw:
        return None
    if isinstance(raw[0], dict):
        df = pd.DataFrame(raw)
        # BingX swap klines returns "time" (not "t") for the timestamp column
        key_map = {"time": "timestamp", "t": "timestamp",
                   "o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"}
        df = df.rename(columns={k: v for k, v in key_map.items() if k in df.columns})
    else:
        # Array format may have extra trailing columns — take only the first 6
        cols = ["timestamp", "open", "high", "low", "close", "volume"]
        df = pd.DataFrame([row[:6] for row in raw], columns=cols)
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["timestamp"] = pd.to_datetime(
        pd.to_numeric(df["timestamp"], errors="coerce"), unit="ms", errors="coerce")
    df = (df.dropna(subset=["timestamp", "close"])
            .sort_values("timestamp")
            .reset_index(drop=True))
    return df if len(df) >= 50 else None

def get_price():
    r = api_get("/openApi/swap/v2/quote/premiumIndex", {"symbol": SYMBOL})
    if r.get("code") == 0 and r.get("data"):
        return float(r["data"]["markPrice"])
    df = get_klines()
    return float(df.iloc[-1]["close"]) if df is not None else None

def get_balance():
    r = api_get("/openApi/swap/v2/user/balance", {"currency": "USDT"}, signed=True)
    if r.get("code") == 0:
        try:
            return float(r["data"]["balance"]["availableMargin"])
        except (KeyError, TypeError):
            # formato alternativo
            try:
                return float(r["data"]["availableMargin"])
            except (KeyError, TypeError):
                pass
    print(f"  ❌ Balance: {r.get('msg', r)}")
    return 0.0

def has_open_position():
    r = api_get("/openApi/swap/v2/user/positions", {"symbol": SYMBOL}, signed=True)
    if r.get("code") != 0:
        return False
    return any(abs(float(p.get("positionAmt", 0))) > 0 for p in r.get("data", []))

def set_leverage():
    for side in ("LONG", "SHORT"):
        r = api_post("/openApi/swap/v2/trade/leverage",
                     {"symbol": SYMBOL, "side": side, "leverage": LEVERAGE})
        ok = r.get("code") == 0
        already = str(r.get("code")) in ("80012", "109107")
        print(f"  {'✅' if ok or already else '❌'} Leverage {LEVERAGE}x {side}"
              f"{' (ya seteado)' if already else ' | ' + r.get('msg', '') if not ok else ''}")

print('✅ Celda 2 OK')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 3 — Indicadores + Señales + Órdenes       ║
# ║            Cipher B (WaveTrend) + Stoch RSI      ║
# ╚══════════════════════════════════════════════════╝

_bars_since_exit = [999]

# ── Indicadores ─────────────────────────────────────
def calculate_indicators(df):
    c  = df["close"]
    hi = df["high"]
    lo = df["low"]

    # ── Cipher B / WaveTrend Oscillator ────────────
    hlc3 = (hi + lo + c) / 3
    esa  = hlc3.ewm(span=WT_N1, adjust=False).mean()
    d    = (hlc3 - esa).abs().ewm(span=WT_N1, adjust=False).mean()
    # protección división por cero
    d_safe = d.where(d != 0, other=np.nan)
    ci   = (hlc3 - esa) / (0.015 * d_safe)
    tci  = ci.fillna(0).ewm(span=WT_N2, adjust=False).mean()
    df["wt1"] = tci
    df["wt2"] = df["wt1"].rolling(4).mean()
    # cruces
    df["wt_cross_bull"] = (df["wt1"] > df["wt2"]) & (df["wt1"].shift(1) <= df["wt2"].shift(1))
    df["wt_cross_bear"] = (df["wt1"] < df["wt2"]) & (df["wt1"].shift(1) >= df["wt2"].shift(1))

    # ── Stochastic RSI ──────────────────────────────
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(STOCH_RSI_LEN).mean()
    loss  = (-delta.clip(upper=0)).rolling(STOCH_RSI_LEN).mean()
    # protección división por cero
    rsi = 100 - 100 / (1 + gain / loss.where(loss != 0, other=np.nan))
    rsi = rsi.fillna(50)
    df["rsi"] = rsi

    rsi_min = rsi.rolling(STOCH_LEN).min()
    rsi_max = rsi.rolling(STOCH_LEN).max()
    rng     = (rsi_max - rsi_min).where((rsi_max - rsi_min) != 0, other=np.nan)
    stoch_raw      = ((rsi - rsi_min) / rng * 100).fillna(50)
    df["stoch_k"]  = stoch_raw.rolling(STOCH_K).mean()
    df["stoch_d"]  = df["stoch_k"].rolling(STOCH_D).mean()

    # ── Volumen ─────────────────────────────────────
    df["vol_ma"] = df["volume"].rolling(20).mean()

    return df

# ── Señal ────────────────────────────────────────────
def get_signal(df):
    r     = df.iloc[-2]   # vela confirmada
    r_ant = df.iloc[-3]   # vela anterior
    px    = float(df.iloc[-1]["close"])
    ts    = r["timestamp"].strftime("%H:%M") if pd.notna(r["timestamp"]) else "--:--"

    # ── Cipher B ────────────────────────────────────
    wt_bull = bool(r["wt_cross_bull"])   # WT1 cruzó arriba WT2 en vela confirmada
    wt_bear = bool(r["wt_cross_bear"])   # WT1 cruzó abajo WT2

    # zona OS/OB: verificamos las últimas 3 velas por si el cruce fue reciente
    recent_os = (df.iloc[-4:-1]["wt2"] <= WT_OS).any()   # zona sobreventa reciente
    recent_ob = (df.iloc[-4:-1]["wt2"] >= WT_OB).any()   # zona sobrecompra reciente

    # ── Stochastic RSI ──────────────────────────────
    sk = r["stoch_k"]
    sd = r["stoch_d"]
    sk_ant = r_ant["stoch_k"]

    stoch_os  = sk < STOCH_OS              # K en sobreventa
    stoch_ob  = sk > STOCH_OB              # K en sobrecompra
    stoch_kup = sk > sd                    # K por encima de D → momentum alcista
    stoch_kdn = sk < sd                    # K por debajo de D → momentum bajista
    # cruce de K sobre D (señal más temprana)
    k_cross_up = sk > sd and sk_ant <= r_ant["stoch_d"]
    k_cross_dn = sk < sd and sk_ant >= r_ant["stoch_d"]

    # ── Volumen ──────────────────────────────────────
    vol_ok = r["volume"] > r["vol_ma"] * VOL_MULT

    # ── Cooldown ─────────────────────────────────────
    cd = _bars_since_exit[0] >= COOLDOWN

    # ── Log ──────────────────────────────────────────
    wt_zone = f"OS({r['wt2']:.1f})" if r["wt2"] <= WT_OS else (
              f"OB({r['wt2']:.1f})" if r["wt2"] >= WT_OB else f"({r['wt2']:.1f})")
    print(f"  [{ts}] WT1={r['wt1']:.1f} WT2={wt_zone} "
          f"StochK={sk:.1f} D={sd:.1f} "
          f"Vol={'✔' if vol_ok else '✘'} CD={'✔' if cd else '✘'}")

    # ── Condiciones de entrada ────────────────────────
    # LONG: Cipher B cruce alcista desde zona OS + StochRSI en sobreventa con K↑
    long_sig = (
        wt_bull and
        recent_os and
        stoch_os and
        stoch_kup and
        vol_ok and
        cd
    )

    # SHORT: Cipher B cruce bajista desde zona OB + StochRSI en sobrecompra con K↓
    short_sig = (
        wt_bear and
        recent_ob and
        stoch_ob and
        stoch_kdn and
        vol_ok and
        cd
    )

    print(f"  🟢 LONG  {'✅' if long_sig else '─'}"
          f" [WT_cruce={'✔' if wt_bull else '✘'}"
          f" OS_reciente={'✔' if recent_os else '✘'}"
          f" StochOS={'✔' if stoch_os else '✘'}"
          f" K↑={'✔' if stoch_kup else '✘'}"
          f" Vol={'✔' if vol_ok else '✘'}]")
    print(f"  🔴 SHORT {'✅' if short_sig else '─'}"
          f" [WT_cruce={'✔' if wt_bear else '✘'}"
          f" OB_reciente={'✔' if recent_ob else '✘'}"
          f" StochOB={'✔' if stoch_ob else '✘'}"
          f" K↓={'✔' if stoch_kdn else '✘'}"
          f" Vol={'✔' if vol_ok else '✘'}]")

    return long_sig, short_sig, px

# ── Órdenes ──────────────────────────────────────────
def calc_qty(balance, price):
    raw = (balance * CAPITAL_PCT / 100 * LEVERAGE) / price
    return int(max(1, round(raw)))

def open_order(side, pos_side, qty):
    r = api_post("/openApi/swap/v2/trade/order", {
        "symbol": SYMBOL, "side": side, "positionSide": pos_side,
        "type": "MARKET", "quantity": qty
    })
    print(f"  {'✅' if r.get('code') == 0 else '❌'} {side}/{pos_side} qty={qty} {r.get('msg', '')}")
    return r

def place_tp(pos_side, qty, entry):
    close_side = "SELL" if pos_side == "LONG" else "BUY"
    tp = round(entry * (1 + TP_PCT) if pos_side == "LONG" else entry * (1 - TP_PCT), 7)
    r  = api_post("/openApi/swap/v2/trade/order", {
        "symbol": SYMBOL, "side": close_side, "positionSide": pos_side,
        "type": "TAKE_PROFIT_MARKET", "quantity": qty,
        "stopPrice": tp, "workingType": "MARK_PRICE"
    })
    print(f"  {'✅' if r.get('code') == 0 else '❌'} TP@{tp:.7f} {r.get('msg', '')}")
    return tp

def place_sl(pos_side, qty, sl_price):
    close_side = "SELL" if pos_side == "LONG" else "BUY"
    r = api_post("/openApi/swap/v2/trade/order", {
        "symbol": SYMBOL, "side": close_side, "positionSide": pos_side,
        "type": "STOP_MARKET", "quantity": qty,
        "stopPrice": sl_price, "workingType": "MARK_PRICE"
    })
    oid = r.get("data", {}).get("order", {}).get("orderId")
    print(f"  {'✅' if r.get('code') == 0 else '❌'} SL@{sl_price:.7f} {r.get('msg', '')}")
    return oid

def cancel_order(oid):
    if oid:
        api_delete("/openApi/swap/v2/trade/order", {"symbol": SYMBOL, "orderId": oid})

# ── Trailing Stop ─────────────────────────────────────
T = {"on": False, "active": False, "ps": None, "entry": None,
     "qty": None, "sl_id": None, "peak": None, "sl": None}

def reset_trail():
    T.update({"on": False, "active": False, "ps": None, "entry": None,
              "qty": None, "sl_id": None, "peak": None, "sl": None})

def _trail_loop():
    while T["on"]:
        time.sleep(SLEEP_TRAIL)
        if not has_open_position():
            print("  📭 Posición cerrada")
            _bars_since_exit[0] = 0
            reset_trail()
            return
        px = get_price()
        if not px:
            continue
        ps, entry = T["ps"], T["entry"]
        if not T["active"]:
            hit = ((ps == "LONG"  and px >= entry * (1 + TRAIL_ACTIVATE_PCT)) or
                   (ps == "SHORT" and px <= entry * (1 - TRAIL_ACTIVATE_PCT)))
            if hit:
                nsl = round(px * (1 - TRAIL_CALLBACK_PCT) if ps == "LONG"
                            else px * (1 + TRAIL_CALLBACK_PCT), 7)
                print(f"  🔄 Trail ON px={px:.6f} SL={nsl:.6f}")
                cancel_order(T["sl_id"])
                T["sl_id"] = place_sl(ps, T["qty"], nsl)
                T.update({"active": True, "peak": px, "sl": nsl})
        else:
            better = ((ps == "LONG" and px > T["peak"]) or
                      (ps == "SHORT" and px < T["peak"]))
            if better:
                T["peak"] = px
                nsl = round(px * (1 - TRAIL_CALLBACK_PCT) if ps == "LONG"
                            else px * (1 + TRAIL_CALLBACK_PCT), 7)
                moved = ((ps == "LONG" and nsl > T["sl"]) or
                         (ps == "SHORT" and nsl < T["sl"]))
                if moved:
                    print(f"  📈 Trail {T['sl']:.6f} → {nsl:.6f}")
                    cancel_order(T["sl_id"])
                    T["sl_id"] = place_sl(ps, T["qty"], nsl)
                    T["sl"] = nsl

def start_trail(ps, entry, qty, sl_id, sl):
    T.update({"on": True, "active": False, "ps": ps, "entry": entry,
              "qty": qty, "sl_id": sl_id, "peak": entry, "sl": sl})
    threading.Thread(target=_trail_loop, daemon=True).start()

print('✅ Celda 3 OK')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 4 — INICIAR BOT  ▶                        ║
# ║  Detener: botón ■ o Runtime → Interrupt          ║
# ╚══════════════════════════════════════════════════╝

def run_bot():
    print("═" * 55)
    print("🤖  PENGU SCALPING BOT v3 — BingX Futuros")
    print(f"    {LEVERAGE}x | TP {TP_PCT*100:.1f}% | SL {SL_PCT*100:.1f}% | R:R {TP_PCT/SL_PCT:.1f}:1")
    print(f"    Cipher B: OS≤{WT_OS} OB≥{WT_OB}")
    print(f"    StochRSI: OS<{STOCH_OS}  OB>{STOCH_OB}")
    print(f"    Trail +{TRAIL_ACTIVATE_PCT*100:.1f}% | callback {TRAIL_CALLBACK_PCT*100:.1f}%")
    print("═" * 55)

    if not detect_symbol():
        return
    set_leverage()
    reset_trail()
    cycle = 0

    while True:
        try:
            cycle += 1
            print(f"\n{'─' * 55}")
            print(f"⏱  Ciclo {cycle} — {datetime.now().strftime('%H:%M:%S')}")

            if has_open_position():
                print("📌 Posición abierta — trailing activo...")
                time.sleep(SLEEP_MAIN)
                continue

            _bars_since_exit[0] = min(_bars_since_exit[0] + 1, 999)

            df = get_klines()
            if df is None:
                print("  ⚠️ Sin datos — reintentando en 60s")
                time.sleep(60)
                continue

            df = calculate_indicators(df)
            long_sig, short_sig, price = get_signal(df)

            balance = get_balance()
            print(f"  💰 Balance: {balance:.2f} USDT")
            if balance < 5:
                print("  ⚠️ Balance insuficiente")
                time.sleep(SLEEP_MAIN)
                continue

            qty = calc_qty(balance, price)

            if long_sig and not short_sig:
                print(f"  🟢 LONG qty={qty} @ {price:.6f}")
                r = open_order("BUY", "LONG", qty)
                if r.get("code") == 0:
                    entry = get_price() or price
                    isl   = round(entry * (1 - SL_PCT), 7)
                    place_tp("LONG", qty, entry)
                    sl_id = place_sl("LONG", qty, isl)
                    start_trail("LONG", entry, qty, sl_id, isl)
                    _bars_since_exit[0] = 0

            elif short_sig and not long_sig:
                print(f"  🔴 SHORT qty={qty} @ {price:.6f}")
                r = open_order("SELL", "SHORT", qty)
                if r.get("code") == 0:
                    entry = get_price() or price
                    isl   = round(entry * (1 + SL_PCT), 7)
                    place_tp("SHORT", qty, entry)
                    sl_id = place_sl("SHORT", qty, isl)
                    start_trail("SHORT", entry, qty, sl_id, isl)
                    _bars_since_exit[0] = 0

            else:
                print("  ⏳ Sin señal")

            time.sleep(SLEEP_MAIN)

        except KeyboardInterrupt:
            print("\n⛔ Bot detenido — revisá posiciones en BingX")
            T["on"] = False
            break
        except Exception as e:
            print(f"  ❌ {e}")
            time.sleep(60)

run_bot()